In [ ]:
import pandas as pd
import datetime as dt
import os

In [ ]:
df = pd.read_csv(r"C:\Users\rovshan\Desktop\Customers\Exxon\Next day prediction\Comprehensive predictions\Operation Level Predictions\Data\Master_Data_With_ID.csv")

In [ ]:
df.head(2)

In [ ]:
# Parse to datetime
df["Start_Time"] = pd.to_datetime(df["Start_Time"], errors="coerce")
df["Job_Report_Start_Date"] = pd.to_datetime(df["Job_Report_Start_Date"], errors="coerce")

# Sort so "first entry per well" is well-defined
df = df.sort_values(["Well_Name", "Start_Time"]).reset_index(drop=True)

# Rows that need fixing: Job_Report_Start_Date year is not 2025 or 2026 (e.g. 1970 defaults, NaT)
year = df["Job_Report_Start_Date"].dt.year
bad = ~year.isin([2025, 2026])

# Standard rule for bad rows:
#   Start_Time hour >= 6  -> same calendar day at 06:00
#   Start_Time hour <  6  -> previous calendar day at 06:00
st = df["Start_Time"]
same_day_6am = st.dt.normalize() + pd.Timedelta(hours=6)
prev_day_6am = same_day_6am - pd.Timedelta(days=1)
six_am_rule  = same_day_6am.where(st.dt.hour >= 6, prev_day_6am)

# Exception: for each well, every row before the first 6AM boundary after the
# crew started shares the well's first Start_Time. "First boundary" is:
#   first Start_Time hour <  6  -> 06:00 same day as first Start_Time
#   first Start_Time hour >= 6  -> 06:00 next day
first_st           = df.groupby("Well_Name")["Start_Time"].transform("first")
first_st_day_6am   = first_st.dt.normalize() + pd.Timedelta(hours=6)
first_boundary     = first_st_day_6am.where(first_st.dt.hour < 6,
                                            first_st_day_6am + pd.Timedelta(days=1))
in_initial_period  = st < first_boundary

replacement = six_am_rule.where(~in_initial_period, first_st)

df.loc[bad, "Job_Report_Start_Date"] = replacement[bad]

print(f"Fixed {bad.sum():,} rows out of {len(df):,}")


In [ ]:
# Sanity check: no Job_Report_Start_Date outside 2025/2026 should remain
bad_after = ~df["Job_Report_Start_Date"].dt.year.isin([2025, 2026])
print(f"Remaining off-year rows: {bad_after.sum()}")
df.loc[bad_after, ["Well_Name", "Job_Report_Start_Date", "Start_Time"]].head()


In [ ]:
df.to_csv("Master_Data_With_ID_Dates_Fixed.csv")